# 病例复杂度调用样例

本笔记本将演示：

1. 如何调用朝厚云 API 生成病例复杂度分析结果。

使用本笔记本前，您需要为病例准备以下材料：

1. 上下颌口腔扫描网格模型。

In [1]:
# 导入必要的包以及定义函数
import os
import time
import requests
import json
import trimesh
import urllib
import numpy as np

# 定义调用规则

请根据您从我方获取的信息修改以下代码块

In [2]:
# 朝厚服务请求地址，随api文档发送
base_url = "<服务请求地址>"

# 朝厚文件服务地址，随api文档发送
file_server_url = "<服务文件服务器地址>"

# 必须传入鉴权 Header。请保护好TOKEN!!! 如果泄露请立即联系我们重置，所有使用该TOKEN的任务都会向您的账户计费
zh_token = "<贵司服务Token, 随合同发送>" # 调用所有的API都必须传入token用作鉴权

user_group = "APIClient" # 用户组，一般为 APIClient

# 贵司user_id, 随api文档发送
user_id = "<贵司user_id>"

# 如果您收到了creds.json, 下面将直接读取
if os.path.exists('../../creds.json'):
    creds = json.load(open('../../creds.json', 'r'))
    base_url = creds['base_url']
    file_server_url = creds['file_server_url']
    zh_token = creds['zh_token']
    user_id = creds['user_id']
    print("loaded creds from creds.json")

loaded creds from creds.json


In [3]:
def upload_file(file_name):
    ext = file_name.split('.')[-1]
    data = open('../../data/' + file_name, 'rb').read()
    resp = requests.get(file_server_url + f"/scratch/{user_group}/{user_id}/upload_url?" +
                        f"postfix={ext}", # 必须指定 postfix, 即文件后缀名
                        headers={"X-ZH-TOKEN": zh_token}) # 获取带签名的上传地址
    resp.raise_for_status()

    upload_url = resp.text[1:-1] # 返回为一个单字符串JSON "string", 这里也可以用json.loads(resp.text)

    resp = requests.put(upload_url, data) # 上传至云储存服务不需要带鉴权头

    resp.raise_for_status()
    path = "/".join(urllib.parse.urlparse(upload_url).path.lstrip("/").split("/")[3:])
    urn = f"urn:zhfile:o:s:{user_group}:{user_id}:{path}"
    return urn

def run_job_and_get_results(json_call, timeout_sec):
    headers = {
      "Content-Type": "application/json",
      "X-ZH-TOKEN": zh_token
    }

    url = base_url + '/run'

    response = requests.request("POST", url, headers=headers, data=json.dumps(json_call))
    print(response.text)
    response.raise_for_status()
    create_result = response.json()
    run_id = create_result['run_id']
    print("workflow id is", run_id)
    url = base_url + f"/run/{run_id}"

    start_time = time.time()
    while time.time()-start_time < timeout_sec:
        time.sleep(0.3)
        response = requests.request("GET", url, headers=headers)
        result = response.json()
        if result['completed'] or result['failed']:
            break

    if not result['completed']:
        if result['failed']:
            raise ValueError("API failed due to " + str(result['reason_public']))
        raise TimeoutError("API timeout")

    print("API finished in {}s".format(time.time()-start_time))
    url = base_url + f"/data/{run_id}"
    response = requests.request("GET", url, headers=headers)
    print(response.text)
    return response.json()

def retrieve_data(urn):
    return requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": urn}),
                        headers={"X-ZH-TOKEN": zh_token}).content

def retrieve_mesh(mesh_file_json):
    resp = requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": mesh_file_json['data']}),
                        headers={"X-ZH-TOKEN": zh_token})
    return trimesh.load(trimesh.util.wrap_as_stream(resp.content), file_type=mesh_file_json['type'])

##  病例复杂度

该章节多个调用可以使用以下单一调用代替

```python
json_call = {
  "spec_group": "mesh-processing",
  "spec_name": "case-complexity-analysis", 
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id
}
```

### 获取分牙结果

In [4]:
# 上颌
json_call = {
  "spec_group": "mesh-processing",
  "spec_name": "oral-denoise-prod",
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
      "mesh": {"type":"drc", "data": upload_file("upper_jaw_scan.drc")},
      "jaw_type": "Upper"
  },
  'output_config': {
    "teeth_comp": {"type": "ply"}
  }
}
result_upper_jaw = run_job_and_get_results(json_call, 300)

{"run_id":"wf_1767166293-d9aa3515-1df3-4f0d-a6f3-d768a995e1b3"}
workflow id is wf_1767166293-d9aa3515-1df3-4f0d-a6f3-d768a995e1b3
API finished in 92.60379815101624s
{"align_matrix":[[-0.9999824860093817,0.0013930166158555263,-0.005752145617455959,-0.07794359018576456],[0.0013012955020852588,0.9998724439575386,0.015918619424891957,0.7105989338378093],[0.005773586797885697,0.015910855385121613,-0.9998567449271925,10.880009664494274],[0.0,0.0,0.0,1.0]],"axis":{"11":[[0.9945672154426575,0.10347811877727509,-0.011329133063554764,-2.0781807381820445],[0.06152930110692978,-0.4965904653072357,0.8658013939857483,-6.900398121230407],[0.08396556228399277,-0.8617947697639465,-0.500259518623352,-23.17791454560653],[0.0,0.0,0.0,1.0]],"12":[[0.9716734290122986,0.23198378086090088,0.045102376490831375,-5.747987922009041],[0.053860023617744446,-0.4032025635242462,0.913524329662323,-3.5477895080689823],[0.23010824620723724,-0.88521808385849,-0.40427589416503906,-22.50366175636299],[0.0,0.0,0.0,1.0]],"13

In [6]:
# 下颌
json_call = {
  "spec_group": "mesh-processing",
  "spec_name": "oral-denoise-prod",
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
      "mesh": {"type":"ply", "data": upload_file("lower_jaw_scan.ply")},
      "jaw_type": "Lower"
  },
  'output_config': {
    "teeth_comp": {"type": "ply"}
  }
}
result_lower_jaw = run_job_and_get_results(json_call, 300)

{"run_id":"wf_1767166428-53894dcb-fd81-41d5-83cb-f5e00bbf63a1"}
workflow id is wf_1767166428-53894dcb-fd81-41d5-83cb-f5e00bbf63a1
API finished in 96.77817225456238s
{"align_matrix":[[-0.9994267717390175,0.029880804920713183,-0.015914315210390564,-0.590816482860842],[0.0309267468548837,0.9970581176025164,-0.07013306247341156,2.2966619735843103],[0.013771864808344917,-0.07058503821785274,-0.997410691801268,12.330294463655179],[0.0,0.0,0.0,1.0]],"axis":{"31":[[-0.9929003119468689,-0.11893949657678604,-0.001555348513647914,-3.905210168456343],[0.036693111062049866,-0.2938216030597687,-0.9551556706428528,-2.483067620202031],[0.1131487563252449,-0.9484314322471619,0.29609981179237366,-18.615781595216795],[0.0,0.0,0.0,1.0]],"32":[[-0.9897286891937256,0.1428850144147873,-0.004583492409437895,-4.880707453175587],[-0.019305076450109482,-0.1653519868850708,-0.9860456585884094,-0.306977673013696],[-0.1416490375995636,-0.9758291840553284,0.16641201078891754,-18.838436329357737],[0.0,0.0,0.0,1.0]],"

### 获取目标定位结果

In [7]:

json_call = {
  "spec_group": "mesh-processing",
  "spec_name": "auto-arrange", 
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data":{
      "upper_teeth_dict": result_upper_jaw["teeth_comp"],
      "upper_axis_matrix_dict": result_upper_jaw["axis"],
      "lower_teeth_dict": result_lower_jaw["teeth_comp"],
      "lower_axis_matrix_dict": result_lower_jaw["axis"],

  },
}
# result_upper_jaw["teeth_comp"]
# result_upper_jaw["axis"]
# #result_lower_jaw["teeth_comp"]
# #result_lower_jaw["axis"]
result_arrangement = run_job_and_get_results(json_call, 500)

{"run_id":"sa_service_1767166533-e763f13a-6381-4cc8-974f-5f33f78c9840"}
workflow id is sa_service_1767166533-e763f13a-6381-4cc8-974f-5f33f78c9840
API finished in 11.055371522903442s
{"result":{"transformation_dict":{"21":[[0.9946658775999533,0.08225391978893445,0.06224214502787105,2.259879665242643],[-0.0934126197058099,0.9742229192195787,0.20533822378451594,1.4129137443588888],[-0.043747850438767076,-0.2100571263886035,0.9767098490520865,-4.571556584180807],[0.0,0.0,0.0,1.0]],"22":[[0.949211011001433,0.31116688449376634,0.04662216841890137,5.840334353634685],[-0.3145781978220059,0.9414892072264692,0.12099020675713187,-3.3720400174587972],[-0.006246122693017017,-0.1295115541969816,0.9915582399843118,-1.9991335718406402],[0.0,0.0,0.0,1.0]],"23":[[0.9945863975473531,0.05233530146774356,-0.08977145444994632,-0.10234014621732612],[-0.041662116028081744,0.9922696155872271,0.1168985802755495,0.4313958539484677],[0.0951954090376266,-0.11252566908335554,0.989078261764711,-0.1398861457568259],[

### 病例复杂度

In [8]:
json_call = {
  "spec_group": "mesh-processing",
  "spec_name": "case-complexity-analysis", 
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data":{
      "teeth_dict": {**result_upper_jaw["teeth_comp"], **result_lower_jaw["teeth_comp"]},
      "axis_dict": {**result_upper_jaw["axis"], **result_lower_jaw["axis"]},
      "transformation_dict": result_arrangement["result"]["transformation_dict"],
      "landmarks_dict": {**result_upper_jaw["landmarks"], **result_lower_jaw["landmarks"]},
      "target_out_form": "" # 暂无对应功能，传入空字符串
  },

}
result_case_complexity= run_job_and_get_results(json_call, 500)

{"run_id":"sa_service_1767167254-0b37e775-5c91-4f08-97bb-97801629a587"}
workflow id is sa_service_1767167254-0b37e775-5c91-4f08-97bb-97801629a587
API finished in 15.982057571411133s
{"result":{"upper_crowding_per_arch":2.025920260595825,"status":{"message":"Some results failed to be calculated. Please check the details","details":{"lower_crowding_per_arch":{"missing_teeth":[43,83],"expected_teeth":[31,32,33,34,35,36,41,42,43,44,45,46,71,72,73,74,75,81,82,83,84,85]},"lower_spacing_per_arch":{"missing_teeth":[43,83],"expected_teeth":[31,32,33,34,35,36,41,42,43,44,45,46,71,72,73,74,75,81,82,83,84,85]},"right_class_2_discrepancy":{"missing_teeth":[43,83],"expected_teeth":[13,43,44,53,83,84]},"right_class_3_discrepancy":{"missing_teeth":[43,83],"expected_teeth":[13,43,44,53,83,84]}},"code":"0100"},"anterior_intrusion_per_tooth":{"11":-0.0712079485733884,"12":0.8386089404261535,"13":0.07934565663480408,"21":-0.029834763224511347,"22":0.39450641286857363,"23":0.0013672268534862398,"31":1.3895

In [9]:
print("=== Case Complexity Analysis Results ===")
print(json.dumps(result_case_complexity, indent=4, ensure_ascii=False))

=== Case Complexity Analysis Results ===
{
    "result": {
        "upper_crowding_per_arch": 2.025920260595825,
        "status": {
            "message": "Some results failed to be calculated. Please check the details",
            "details": {
                "lower_crowding_per_arch": {
                    "missing_teeth": [
                        43,
                        83
                    ],
                    "expected_teeth": [
                        31,
                        32,
                        33,
                        34,
                        35,
                        36,
                        41,
                        42,
                        43,
                        44,
                        45,
                        46,
                        71,
                        72,
                        73,
                        74,
                        75,
                        81,
                        82,
                   